## Non-transitive Games

### Payoff Matrix

Non-transitive component of games can be represented by a skew-symmetric payoff matrix. For games where probabilities of a player winning over another is dependent on their individual abilities and a non-transitive payoffs

In [89]:
library(tidyverse)
library(future.apply)
library(dplyr)
library(ggplot2)
library(patchwork)
library(latex2exp)
library(see)
library(expm)
options(warn = -1)

Loading required package: Matrix

Attaching package: ‘Matrix’

The following objects are masked from ‘package:tidyr’:

    expand, pack, unpack


Attaching package: ‘expm’

The following object is masked from ‘package:Matrix’:

    expm



In [8]:
simulate_elo_game <- function(thetas,N_opts=3){
  t_ids <- sample(c(1:length(thetas)),2,replace=FALSE)
  t <- thetas[t_ids]
  dt <- t[1]-t[2]
  p <- 1/(1+exp(-dt))
  w <- rbinom(n=1,size=1,prob=p)
  ## Hand played is randomly chosen
  h <- sample(c(1:N_opts),2,replace=TRUE)
  res <- c(t_ids[1],t_ids[2],h[1],h[2],w,p)
  return(res)
}

## Non-transitive payoff probability matrix (eps -> [0,2])
payoff_matrix <- function(N,eps=0) {
  eps <- abs(eps)
  # generate square matrix with 0s
  
  A <- matrix(0, nrow = N, ncol = N)
  
  # Fill upper triangle with payoff
  for(i in c(1:(N-1))){
    A[i,(i+1):N] <- rep(c(-1+eps, 1-eps), length.out = (N-i))
  }
  
  # Enforce skew-symmetry for cyclic game
  A <- A - t(A)
  
  # Convert to Probabilities
  pA <- 0.5*(A + 1)
  return(pA)
}


simulate_rps <- function(thetas,N_opts,payoff=NULL,replace=FALSE){
  t_ids <- sample(c(1:length(thetas)),2,replace=replace)
  if(is.null(payoff)){
    p <- payoff_matrix(N_opts,eps=0)
  }else{
    p <- payoff
  }
  
  h <- sample(c(1:N_opts),2,replace=replace)
  w <- rbinom(1,1,p[h[1],h[2]])
  # w <- ifelse(prob>0,rbinom(1,1,prob),rbinom(1,1,1+prob))
  res <- c(t_ids[1],t_ids[2],h[1],h[2],w,p[h[1],h[2]])
  return(res)
}

simulate_rps_luck <- function(thetas,N_opts,payoff=NULL,replace=FALSE){
  t_ids <- sample(c(1:length(thetas)),2,replace=replace)
  
  if(is.null(payoff)){
    p <- payoff_matrix(N_opts,eps=0)
  }else{
    p <- payoff
  }
  idx <- which(p>0.5,arr.ind = TRUE,useNames = FALSE)
  rps_wins <- list()
  for (i in c(1:nrow(idx))){
    rps_wins[[i]] <- c(idx[i,1],idx[i,2])
  }
  
  h <- sample(rps_wins,1)[[1]]
  
  ## Luckier player to win
  dt <- thetas[t_ids[1]] - thetas[t_ids[2]]
  w <- rbinom(1,1,plogis(dt))
  if(w==0){
    h <- c(h[2],h[1])
  }
  res <- c(t_ids[1],t_ids[2],h[1],h[2],w,w)
  return(res)
}

simulate_NT_model <- function(thetas,phis,N_opts,replace=FALSE){
  t_ids <- sample(c(1:length(thetas)),2,replace=FALSE)
  phi_ids <- sample(c(1:length(phis)),2,replace=FALSE)
  h <- sample(c(1:N_opts),2,replace=replace)
  
  d_theta <- thetas[t_ids[1]] - thetas[t_ids[2]]
  d_phi <- phis[h[1]] - phis[h[2]]
  d_phi2 <- atan2(sinpi(d_phi),cospi(d_phi)) %% (2*pi)
  g_d_phi <- 2*sin(d_phi2)
  # g_d_phi <- 2*atanh(sinpi(d_phi))
  
  logit <- d_theta + g_d_phi 
  
  p <- plogis(logit)
  w <- rbinom(1,1,p)
  
  res <- c(t_ids[1],t_ids[2],h[1],h[2],w,p)
  return(res)
}

simulate_dice_rolls <- function(thetas,dice,replace=FALSE){
  t_ids <- sample(c(1:length(thetas)),2,replace=FALSE)
  N_opts <- length(dice)
  
  h <- sample(c(1:N_opts),2,replace=replace)
  roll_1 <- sample(dice[[h[1]]],1)
  roll_1_w_handicap <- roll_1 + thetas[t_ids[1]]
  roll_2 <- sample(dice[[h[2]]],1)
  roll_2_w_handicap <- roll_2 + thetas[t_ids[2]]
  
  p <- mean(outer(dice[[h[1]]]+ thetas[t_ids[1]],dice[[h[2]]]+ thetas[t_ids[2]],">="))
  
  w <- ifelse(roll_1_w_handicap >= roll_2_w_handicap,1,0)
  res <- c(t_ids[1],t_ids[2],h[1],h[2],w,p)
  return(res)
}


In [64]:
LogLikELO <- function(data, theta) {
  p <- plogis(theta[data$player1] - theta[data$player2])
  ll <- sum(data$result * log(p) + (1 - data$result) * log(1 - p))
  return(ll)
}

fit_elo_model <- function(N_players,matches){
  
  starting_theta <- rnorm(N_players)

  sol <- optim(
    starting_theta,
    LogLikELO,
    data = matches,
    method = 'L-BFGS-B',
    lower = -3,
    upper = 3,
    control = list(fnscale = -1, maxit = 1e6)
  )

  return(sol)
}


LogLik_NT <- function(params, data, N_ids,
                      theta_centering = TRUE,
                      ridge_penalty = TRUE) {
  
  ## Mean centering or setting first theta to 0
  if (theta_centering) {
      theta <- params[1:N_ids]
      theta <- theta - mean(theta)
      pos <- params[(N_ids + 1):length(params)]
      n_pos <- length(params) - N_ids
  } else {
      theta <- c(0, params[1:(N_ids - 1)])
      pos <- params[N_ids:length(params)]
      n_pos <- length(params) - (N_ids + 1)
  }

  d_theta <- theta[data$player1] - theta[data$player2]
  P <- get_payoff_mat(pos)
  data <- data |> mutate(p=P[cbind(hand1,hand2)])
  d_phi <- qlogis(data$p)
  logit <- d_theta + d_phi

  p <- plogis(logit)
  ll <- sum(data$result * log(p) + (1 - data$result) * log(1 - p))

  penalty <- ifelse(ridge_penalty, 0.1 * sum(theta^2),0)
  neg_ll <- -ll + penalty

  if (is.finite(neg_ll)) neg_ll else 1e10
}

estimate_nt_elo <- function(N_ids, N_opts, matches,
                            theta_centering = TRUE,
                            ridge_penalty = TRUE) {
  ## Initializing params
  n_theta <- ifelse(theta_centering,N_ids,N_ids-1)
  starting_theta <- rnorm(n_theta)

  n_payoff <- N_opts*(N_opts-1)/2
  starting_po <- runif(n_payoff)
  starting_params <- c(starting_theta, starting_po)

  ## Solver
  sol <- optim(
    starting_params,
    LogLik_NT,
    data = matches,
    N_ids = N_ids,
    theta_centering = theta_centering,
    ridge_penalty = ridge_penalty,
    method = 'L-BFGS-B',
    hessian = TRUE,
    lower = c(rep(-Inf,n_theta),rep(0,n_payoff)),
    upper = c(rep(Inf,n_theta),rep(1,n_payoff)),
    control = list(fnscale = 1, maxit = 1e6)
  )
  return(sol)
  }

get_payoff_mat <- function(x){
  n <- (1+sqrt(1+8*length(x)))/2
  P <- matrix(0,n,n)
  k <- 0
  for(i in c(1:(n-1))){
    l_out <- n-i
    P[i,c((i+1):n)] <- x[seq(from=k+1,by=1,length.out=l_out)]
    k <- k + n-i
  }
  P[which(upper.tri(P),arr.ind = TRUE)[,c("col","row")]] <- 1-P[upper.tri(P)]
  diag(P) <- 0.5
  return(P)
  
}

In [65]:
set.seed(1729)
N_ids <- 100
N_games <- 1000
N_opts <- 3 ## For rock-paper-scissors
payoff <- payoff_matrix(N_opts,eps=0)
print(payoff)
rps_thetas <- rep(0,N_ids)

rps_matches <- as.data.frame(do.call(rbind, replicate(N_games,simulate_rps(rps_thetas,N_opts,payoff),simplify = FALSE)))
rps_matches <- rps_matches |> rename(player1="V1",
                             player2="V2",
                             hand1 = "V3",
                             hand2 = "V4",
                             result="V5",
                             true_p="V6")


     [,1] [,2] [,3]
[1,]  0.5  0.0  1.0
[2,]  1.0  0.5  0.0
[3,]  0.0  1.0  0.5


In [66]:
est <- estimate_nt_elo(N_ids,N_opts,rps_matches)
get_payoff_mat(tail(est$par,N_opts*(N_opts-1)/2))

            [,1]         [,2]        [,3]
[1,] 0.500000000 0.0003977144 0.990322183
[2,] 0.999602286 0.5000000000 0.003655781
[3,] 0.009677817 0.9963442188 0.500000000

### Adding some noise to the payoff matrix

In [67]:
set.seed(1729)
N_ids <- 100
N_games <- 10000
N_opts <- 3 ## For rock-paper-scissors
payoff <- payoff_matrix(N_opts,eps=0.2)
print(payoff)
rps_thetas <- rep(0,N_ids)

rps_matches <- as.data.frame(do.call(rbind, replicate(N_games,simulate_rps(rps_thetas,N_opts,payoff),simplify = FALSE)))
rps_matches <- rps_matches |> rename(player1="V1",
                             player2="V2",
                             hand1 = "V3",
                             hand2 = "V4",
                             result="V5",
                             true_p="V6")

     [,1] [,2] [,3]
[1,]  0.5  0.1  0.9
[2,]  0.9  0.5  0.1
[3,]  0.1  0.9  0.5


In [68]:
est <- estimate_nt_elo(N_ids,N_opts,rps_matches)
get_payoff_mat(tail(est$par,N_opts*(N_opts-1)/2))

          [,1]       [,2]       [,3]
[1,] 0.5000000 0.09250567 0.89904895
[2,] 0.9074943 0.50000000 0.08969768
[3,] 0.1009511 0.91030232 0.50000000

### RPS with "luck"

In [69]:
N_ids <- 100
N_opts <- 5
N_games <- 10000
payoff <- payoff_matrix(N_opts,eps=0)
print(payoff)
rps_luck_thetas <- rnorm(N_ids,mean=0,sd=1)
rps_luck_matches <- as.data.frame(do.call(rbind, replicate(N_games,simulate_rps_luck(rps_luck_thetas,N_opts,payoff),simplify = FALSE)))

rps_luck_matches <- rps_luck_matches |> rename(player1="V1",
                             player2="V2",
                             hand1 = "V3",
                             hand2 = "V4",
                             result="V5",
                             true_p="V6")

     [,1] [,2] [,3] [,4] [,5]
[1,]  0.5  0.0  1.0  0.0  1.0
[2,]  1.0  0.5  0.0  1.0  0.0
[3,]  0.0  1.0  0.5  0.0  1.0
[4,]  1.0  0.0  1.0  0.5  0.0
[5,]  0.0  1.0  0.0  1.0  0.5


In [70]:
est <- estimate_nt_elo(N_ids,N_opts,rps_luck_matches)
get_payoff_mat(tail(est$par,N_opts*(N_opts-1)/2))

           [,1]         [,2]        [,3]        [,4]       [,5]
[1,] 0.50000000 0.0006879956 0.975449528 0.001847372 0.98864842
[2,] 0.99931200 0.5000000000 0.008417527 0.982721380 0.02068344
[3,] 0.02455047 0.9915824729 0.500000000 0.008889354 0.99872301
[4,] 0.99815263 0.0172786197 0.991110646 0.500000000 0.01934971
[5,] 0.01135158 0.9793165626 0.001276993 0.980650292 0.50000000

In [71]:
tail(est$par,N_opts*(N_opts-1)/2)

 [1] 0.0006879956 0.9754495281 0.0018473716 0.9886484223 0.0084175271 0.9827213803 0.0206834374 0.0088893538 0.9987230072
[10] 0.0193497081

## Non-transitive dice games

In [72]:
grime_3 <- c(7/12,11/36,7/12) #Red, Blue, Olive
grime_4 <- c(2/3,1/3,4/9,1/2,2/3,1/3) #Red, Blue, Olive, Yellow
grime_5 <- c(7/12,11/36,13/18,4/9,7/12,1/3,2/3,5/9,5/18,5/9) #Red, Blue, Olive, Yellow, Magenta


grime_mats_list <- list(grime_3,grime_4,grime_5)

grime_dice_payoffs <- function(k,grime_mats=grime_mats_list){
  p <- matrix(0,k,k)
  upper_tri_idx <- as.data.frame(which(upper.tri(p),arr.ind = TRUE)) |> 
    arrange(row,col) |> 
    as.matrix()
  
  lower_tri_idx <- as.data.frame(which(lower.tri(p),arr.ind = TRUE)) |> 
    arrange(col,row) |> 
    as.matrix()
  
  p[upper_tri_idx] <- grime_mats[[k-2]]
  p[lower_tri_idx] <- 1-p[upper_tri_idx]
  diag(p) <- 1/2  
  return(p)
}

### 3-dice

In [73]:
N_games <- 1000
N_ids <- 100
N_opts <- 3
thetas <- rnorm(N_ids)

payoff_mat <- grime_dice_payoffs(N_opts)
print(payoff_mat)
grime_dice_matches <- as.data.frame(
  do.call(rbind, 
          replicate(N_games,
                    simulate_rps(thetas,N_opts,payoff_mat,TRUE),
                    simplify = FALSE)))

grime_dice_matches <- grime_dice_matches |> rename(player1="V1",
                             player2="V2",
                             hand1 = "V3",
                             hand2 = "V4",
                             result="V5",
                             true_p="V6")

          [,1]      [,2]      [,3]
[1,] 0.5000000 0.5833333 0.3055556
[2,] 0.4166667 0.5000000 0.5833333
[3,] 0.6944444 0.4166667 0.5000000


In [ ]:
est <- estimate_nt_elo(N_ids,N_opts,grime_dice_matches)
p_dice3 <- get_payoff_mat(tail(est$par,N_opts*(N_opts-1)/2))
p_dice3

          [,1]      [,2]      [,3]
[1,] 0.5000000 0.6630361 0.2942908
[2,] 0.3369639 0.5000000 0.4680732
[3,] 0.7057092 0.5319268 0.5000000

### 4-dice

In [75]:
N_games <- 5000
N_ids <- 100
N_opts <- 4
thetas <- rnorm(N_ids)

payoff_mat <- grime_dice_payoffs(N_opts)
print(payoff_mat)
grime_dice_matches <- as.data.frame(
  do.call(rbind, 
          replicate(N_games,
                    simulate_rps(thetas,N_opts,payoff_mat,TRUE),
                    simplify = FALSE)))

grime_dice_matches <- grime_dice_matches |> rename(player1="V1",
                             player2="V2",
                             hand1 = "V3",
                             hand2 = "V4",
                             result="V5",
                             true_p="V6")

          [,1]      [,2]      [,3]      [,4]
[1,] 0.5000000 0.6666667 0.3333333 0.4444444
[2,] 0.3333333 0.5000000 0.5000000 0.6666667
[3,] 0.6666667 0.5000000 0.5000000 0.3333333
[4,] 0.5555556 0.3333333 0.6666667 0.5000000


In [ ]:
est <- estimate_nt_elo(N_ids,N_opts,grime_dice_matches)
p_dice4 <- get_payoff_mat(tail(est$par,N_opts*(N_opts-1)/2))
p_dice4

          [,1]      [,2]      [,3]      [,4]
[1,] 0.5000000 0.6603741 0.3397652 0.4116249
[2,] 0.3396259 0.5000000 0.5271505 0.6790363
[3,] 0.6602348 0.4728495 0.5000000 0.3179269
[4,] 0.5883751 0.3209637 0.6820731 0.5000000

### 5-dice

In [105]:
N_games <- 10000
N_ids <- 100
N_opts <- 5
thetas <- rnorm(N_ids)

payoff_mat <- grime_dice_payoffs(N_opts)
print(payoff_mat)
grime_dice_matches <- as.data.frame(
  do.call(rbind, 
          replicate(N_games,
                    simulate_rps(thetas,N_opts,payoff_mat,TRUE),
                    simplify = FALSE)))

grime_dice_matches <- grime_dice_matches |> rename(player1="V1",
                             player2="V2",
                             hand1 = "V3",
                             hand2 = "V4",
                             result="V5",
                             true_p="V6")

          [,1]      [,2]      [,3]      [,4]      [,5]
[1,] 0.5000000 0.5833333 0.3055556 0.7222222 0.4444444
[2,] 0.4166667 0.5000000 0.5833333 0.3333333 0.6666667
[3,] 0.6944444 0.4166667 0.5000000 0.5555556 0.2777778
[4,] 0.2777778 0.6666667 0.4444444 0.5000000 0.5555556
[5,] 0.5555556 0.3333333 0.7222222 0.4444444 0.5000000


In [106]:
est_d5 <- estimate_nt_elo(N_ids,N_opts,grime_dice_matches)
p_dice5 <- get_payoff_mat(tail(est_d5$par,N_opts*(N_opts-1)/2))
p_dice5

          [,1]      [,2]      [,3]      [,4]      [,5]
[1,] 0.5000000 0.5695577 0.3098943 0.7478640 0.4505984
[2,] 0.4304423 0.5000000 0.6010121 0.3506020 0.6741813
[3,] 0.6901057 0.3989879 0.5000000 0.5633336 0.2732331
[4,] 0.2521360 0.6493980 0.4366664 0.5000000 0.5828060
[5,] 0.5494016 0.3258187 0.7267669 0.4171940 0.5000000

In [116]:
svd(2*p_dice5 -1)

$d
[1] 9.333788e-01 9.333788e-01 2.209985e-01 2.209985e-01 8.666243e-18

$u
           [,1]          [,2]       [,3]        [,4]        [,5]
[1,] -0.6747730 -1.110223e-16  0.6821055 -0.02328665  0.28084004
[2,]  0.4494872  3.146491e-01  0.1330296 -0.39630816  0.72401671
[3,] -0.1576609 -6.564693e-01 -0.3877682  0.23377977  0.58238645
[4,] -0.1381305  6.206407e-01 -0.2112536  0.70267342  0.23947245
[5,]  0.5465397 -2.912888e-01  0.5674882  0.54221274 -0.02019049

$v
              [,1]       [,2]        [,3]       [,4]        [,5]
[1,]  8.783085e-17 -0.6747730 -0.02328665 -0.6821055 -0.28084004
[2,] -3.146491e-01  0.4494872 -0.39630816 -0.1330296 -0.72401671
[3,]  6.564693e-01 -0.1576609  0.23377977  0.3877682 -0.58238645
[4,] -6.206407e-01 -0.1381305  0.70267342  0.2112536 -0.23947245
[5,]  2.912888e-01  0.5465397  0.54221274 -0.5674882  0.02019049


In [111]:
round(Schur(2*p_dice5 -1)$T,2)

     [,1]  [,2]  [,3] [,4] [,5]
[1,] 0.00 -0.93  0.00 0.00    0
[2,] 0.93  0.00  0.00 0.00    0
[3,] 0.00  0.00  0.00 0.22    0
[4,] 0.00  0.00 -0.22 0.00    0
[5,] 0.00  0.00  0.00 0.00    0

In [113]:
Schur(2*p_dice5 -1)$EValues

[1] 3.014929e-17+0.9333788i 3.014929e-17-0.9333788i 2.782179e-19+0.2209985i 2.782179e-19-0.2209985i 6.046953e-18+0.0000000i

## Starcraft

In [126]:
starcraft <- read.csv("lotv_24-26.csv")

starcraft <- starcraft |> filter(hand1!=4,hand2!=4,sca!=scb) 

N_ids_old <- 1e6
N_ids <- 0
while (N_ids != N_ids_old){
  N_ids_old <- N_ids
  player_counts <- bind_rows(
  starcraft |> select(player = player1),
  starcraft |> select(player = player2)
  ) |>
  count(player, name = "games_played")
  
  # Keep only players with at least 10 games
  valid_players <- player_counts |>
    filter(games_played >= 25) |>
    pull(player)
  
  # Filter matches where BOTH players are valid
  starcraft_filtered <- starcraft |>
    filter(player1 %in% valid_players,
           player2 %in% valid_players)
  
  ids <- unique(sort(c(starcraft_filtered$player1,starcraft_filtered$player2)))
  N_ids <- length(ids)
  N_opts <- length(unique(c(starcraft_filtered$hand1,starcraft_filtered$hand2)))
  starcraft <- starcraft_filtered
  
}

starcraft_filtered <- starcraft_filtered|> mutate(player1 = match(player1,ids),
                                player2 = match(player2,ids)) 

In [ ]:
N_opts <- length(unique(c(starcraft_filtered$hand1,starcraft_filtered$hand2)))
est <- estimate_nt_elo(N_ids,N_opts,starcraft_filtered)
p_starcraft <- get_payoff_mat(tail(est$par,N_opts*(N_opts-1)/2))
p_starcraft

In [ ]:
avg_player_ratings <- rbind(
  starcraft_filtered |> 
    select(player1,rta) |> 
    group_by(player1) |> 
    summarise(theta=mean(rta)) |> 
    rename(player=player1),
  starcraft_filtered |> 
    select(player2,rtb) |> 
    group_by(player2) |> 
    summarise(theta=mean(rtb))|>
    rename(player=player2)) |> 
  group_by(player) |> 
  summarise(theta=mean(theta))

In [123]:
A_starcraft <- 2*p_starcraft - 1
svd_stracraft <- svd(A_starcraft)
svd_stracraft$d

[1] 2.293121e-01 2.293121e-01 1.751540e-17

In [124]:
Schur(A_starcraft)

$Q
           [,1]       [,2]        [,3]
[1,] -0.9021737 -0.2027150  0.38077449
[2,]  0.2071393 -0.9778572 -0.02980953
[3,]  0.3783859  0.0519800  0.92418731

$T
             [,1]          [,2]          [,3]
[1,] 2.859959e-17  5.728817e-17  5.400767e-17
[2,] 0.000000e+00 -2.696951e-17  2.293121e-01
[3,] 0.000000e+00 -2.293121e-01 -2.696951e-17

$EValues
[1]  2.859959e-17+0.0000000i -2.696951e-17+0.2293121i -2.696951e-17-0.2293121i
